# Analzye collected data

Here, we gather the data collected from the different models and compare them to human data.

### Import

In [1]:
import pandas as pd
import plotly.express as px
from pathlib import Path
import  matplotlib.pyplot as plt

## Parameters

In [2]:
MODELS = [
    "gpt-5.4-nano-2026-03-17",
    "gpt-5.4-mini-2026-03-17",
    "gpt-5.4",
]

### Functions

In [3]:
def read_scores_from_model(model):

    model_name = model + "_"
    files = list(Path("results").glob(f"{model_name}*.csv"))

    df = pd.DataFrame([])

    print(f"Gathered data for model {model} from {len(files)} files.")
    for f in files:
        df_data = pd.read_csv(f'results/{f.name}')
        df_data["run"] = f.name[-5:-4]
        df = pd.concat([df, df_data], ignore_index=True)

    df["model"] = model
    df["id"] = df["id"].astype(int)

    return df


def summarize_scores_from_model(df, model):
    summary = df.groupby("id").agg({"score": ["mean", "std", "sem"]}).reset_index()
    summary.columns = summary.columns.get_level_values(1)
    #summary.columns = ['id', f'{model}_mean', f'{model}_std', f'{model}_sem']
    summary.columns = ['id', 'score_mean', 'score_std', 'score_sem']
    print(f"Summarized data for model {model}.")

    return summary


def collate_human_and_llm_data_wide(MODELS: list) -> pd.DataFrame:
    
    # fetch human data
    df = pd.read_csv("data/iCO-Eval2_summarizedRatings.csv", sep=";")

    # fetch, aggregate and merge LLm data
    for model in MODELS:
        df_model = read_scores_from_model(model)
        summary_model = summarize_scores_from_model(df_model, model)
        summary_model.columns = ['id', f'{model}_mean', f'{model}_std', f'{model}_sem']
        df = pd.merge(df, summary_model, left_on="CODE", right_on="id", how="left").drop(columns='id')
        
    return df


def collate_human_and_llm_data_long(MODELS: list) -> pd.DataFrame:
    
    # fetch human data
    df = pd.read_csv("data/iCO-Eval2_summarizedRatings.csv", sep=";")
    df.drop(columns=["CONTEXT QUESTION", "CRITICAL UTTERANCE", "CORRECT RESPONSE (0=no; 1=yes)", "FUN_std", "FUN_sem", "COH_std", "COH_sem", "DIR_std", "DIR_sem", "PRE_std", "PRE_sem", "SSI_std", "SSI_sem", "CER_std", "CER_sem"], inplace=True) #this is not really necessary

    # add information such as condition and set
    ref = df

    # put them on the same scale as the LLm scores (0-1)
    #df["FUN_mean"] = rescale(df["FUN_mean"], 1, 7)
    #df["COH_mean"] = rescale(df["COH_mean"], 1, 7)
    #df["DIR_mean"] = rescale(df["DIR_mean"], 1, 7)
    #df["PRE_mean"] = rescale(df["PRE_mean"], 1, 7)
    #df["SSI_mean"] = rescale(df["SSI_mean"], 1, 7)
    #df["CER_mean"] = rescale(df["CER_mean"], 1, 4)
    df["evaluator"] = "human"

    # fetch, aggregate and merge LLm data
    for model in MODELS:
        df_model = read_scores_from_model(model)
        summary_model = summarize_scores_from_model(df_model, model)
        
        # only take mean, leave sd and std out
        summary_model  = summary_model[["id", "score_mean"]] 
        summary_model.rename(columns={"score_mean": "FUN_mean", "id": "CODE"}, inplace=True)
        
        # generate certainty score
        summary_model["CER_mean"] = ((summary_model["FUN_mean"] - 4).abs())+1
        
        # add evaluator column to indicate which model the scores are from
        summary_model["evaluator"] = model

        # add SET and CONDITION infformation from the reference human data
        summary_model = summary_model.merge(ref[["CODE", "SET (1=SA-matched;2=non-SA-matched)", "CONDITION"]], on="CODE", how="left", validate="m:1")

        # concatenate with data so far
        df = pd.concat([df, summary_model], axis=0, join='outer').reset_index(drop=True)
        
    # rename SET column
    df = df.rename(columns={"SET (1=SA-matched;2=non-SA-matched)": "SET"})

    return df

def rescale(series: pd.Series, min: int, max: int) -> pd.Series:
    return (series - min) / (max - min)

### Aggregation

In [4]:
df_wide =  collate_human_and_llm_data_wide(MODELS)
df_long = collate_human_and_llm_data_long(MODELS)


Gathered data for model gpt-5.4-nano-2026-03-17 from 28 files.
Summarized data for model gpt-5.4-nano-2026-03-17.
Gathered data for model gpt-5.4-mini-2026-03-17 from 28 files.
Summarized data for model gpt-5.4-mini-2026-03-17.
Gathered data for model gpt-5.4 from 28 files.
Summarized data for model gpt-5.4.
Gathered data for model gpt-5.4-nano-2026-03-17 from 28 files.
Summarized data for model gpt-5.4-nano-2026-03-17.
Gathered data for model gpt-5.4-mini-2026-03-17 from 28 files.
Summarized data for model gpt-5.4-mini-2026-03-17.
Gathered data for model gpt-5.4 from 28 files.
Summarized data for model gpt-5.4.


### Visualization

In [5]:
fig = px.strip(df_long, y="FUN_mean", x="evaluator", color="CONDITION", stripmode='group')
fig.update_traces(marker_size=3)
fig.show()

In [15]:
import numpy as np
import plotly.graph_objects as go

x_order = df_long["evaluator"].dropna().unique().tolist()
cond_order = df_long["CONDITION"].dropna().unique().tolist()
x_map = {x: i for i, x in enumerate(x_order)}
offsets = np.linspace(-0.18, 0.18, len(cond_order))
offset_map = dict(zip(cond_order, offsets))
palette = px.colors.qualitative.Plotly
color_map = {cond: palette[i % len(palette)] for i, cond in enumerate(cond_order)}

summary = (
    df_long.groupby(["evaluator", "CONDITION"], as_index=False)["CER_mean"]
    .agg(mean="mean", sd="std")
)

fig = go.Figure()

for cond in cond_order:
    d = df_long[df_long["CONDITION"] == cond]
    fig.add_trace(
        go.Scatter(
            x=[x_map[x] + offset_map[cond] for x in d["evaluator"]],
            y=d["CER_mean"],
            mode="markers",
            name=str(cond),
            legendgroup=str(cond),
            marker=dict(size=3, color=color_map[cond]),
            opacity=1,
        )
    )

for cond in cond_order:
    d = summary[summary["CONDITION"] == cond]
    fig.add_trace(
        go.Scatter(
            x=[x_map[x] + offset_map[cond] for x in d["evaluator"]],
            y=d["mean"],
            mode="markers",
            name=f"{cond} mean ± SD",
            showlegend=False,
            marker=dict(
                size=3,
                symbol="diamond",
                color="black",
                line=dict(width=3, color="black"),
            ),
            error_y=dict(
                type="data",
                array=d["sd"],
                visible=True,
                color="black",
                thickness=1,
                width=6,
            ),
        )
    )

fig.update_layout(
    xaxis=dict(
        tickmode="array",
        tickvals=list(x_map.values()),
        ticktext=x_order,
        title="evaluator",
    ),
    yaxis=dict(title="CER_mean"),
)

fig.show()

## By evaluator, condition and set

In [7]:
fig = px.strip(
    df_long,
    y="FUN_mean",
    x="evaluator",
    color="CONDITION",
    stripmode='group',
    facet_row="SET")
fig.update_traces(marker_size=3)
fig.show()

In [8]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

x_order = df_long["evaluator"].dropna().unique().tolist()
cond_order = df_long["CONDITION"].dropna().unique().tolist()
set_order = df_long["SET"].dropna().unique().tolist()
x_map = {x: i for i, x in enumerate(x_order)}
offsets = np.linspace(-0.18, 0.18, len(cond_order))
offset_map = dict(zip(cond_order, offsets))
palette = px.colors.qualitative.Plotly
color_map = {cond: palette[i % len(palette)] for i, cond in enumerate(cond_order)}

summary = (
    df_long.groupby(["SET", "evaluator", "CONDITION"], as_index=False)["CER_mean"]
    .agg(mean="mean", sd="std")
)

fig = make_subplots(
    rows=len(set_order),
    cols=1,
    shared_xaxes=True,
    subplot_titles=[f"SET={set_value}" for set_value in set_order],
    vertical_spacing=0.08,
)

for row, set_value in enumerate(set_order, start=1):
    df_set = df_long[df_long["SET"] == set_value]
    summary_set = summary[summary["SET"] == set_value]

    for cond in cond_order:
        d = df_set[df_set["CONDITION"] == cond]
        fig.add_trace(
            go.Scatter(
                x=[x_map[x] + offset_map[cond] for x in d["evaluator"]],
                y=d["CER_mean"],
                mode="markers",
                name=str(cond),
                legendgroup=str(cond),
                showlegend=row == 1,
                marker=dict(size=3, color=color_map[cond]),
                opacity=0.5,
            ),
            row=row,
            col=1,
        )

    for cond in cond_order:
        d = summary_set[summary_set["CONDITION"] == cond]
        fig.add_trace(
            go.Scatter(
                x=[x_map[x] + offset_map[cond] for x in d["evaluator"]],
                y=d["mean"],
                mode="markers",
                name=f"{cond} mean ± SD",
                showlegend=False,
                marker=dict(
                    size=6,
                    symbol="diamond",
                    color="black",
                    line=dict(width=0.5, color="black"),
                ),
                error_y=dict(
                    type="data",
                    array=d["sd"],
                    visible=True,
                    color="black",
                    thickness=1,
                    width=3,
                ),
            ),
            row=row,
            col=1,
        )

for row in range(1, len(set_order) + 1):
    fig.update_xaxes(
        tickmode="array",
        tickvals=list(x_map.values()),
        ticktext=x_order,
        title_text="evaluator" if row == len(set_order) else None,
        row=row,
        col=1,
    )
    fig.update_yaxes(title_text="CER_mean", row=row, col=1)

fig.update_layout(height=350 * len(set_order))

fig.show()